# Labyrinth Level

Labyrinth Level turns a level design into a maze a game can load. Every room is a qubit. Neighbouring rooms the design wants open are set to agree, the rest to disagree; after sampling, rooms that agree are joined by a corridor and rooms that disagree get a wall. On hardware, noise reshapes the level, and that's part of the result.

## Setup

In [ ]:
import getpass
import os
import json

import matplotlib.pyplot as plt

from moth_atlas import Atlas

ENGINE = "labyrinth-staging"  # the engine this notebook runs: Atlas V2 (staging). Production: "labyrinth-v1"

if not os.environ.get("MOTH_API_KEY"):
    os.environ["MOTH_API_KEY"] = getpass.getpass("Atlas API key")

client = Atlas()  # reads MOTH_API_KEY, and MOTH_API_BASE for which API to call

In [ ]:
def crop(level, rows, cols):
    """The top-left rows × cols rooms of a level: its rooms, couplings and initial states, renumbered."""
    width = level["grid_size"]["cols"]
    keep = {r * width + c: r * cols + c for r in range(rows) for c in range(cols)}
    return {
        "name": f"{level['name']} ({rows}×{cols})",
        "grid_size": {"rows": rows, "cols": cols},
        "num_qubits": rows * cols,
        "coupling_map": [[keep[a], keep[b]] for a, b in level["coupling_map"] if a in keep and b in keep],
        "initial_states": {str(keep[int(q)]): s for q, s in level["initial_states"].items() if int(q) in keep},
        "relationships": [],
    }


def walls(level, designed=False):
    """Which grid edges are walls: measured (neighbours whose ZZ came out negative) or designed (target sign −1)."""
    if designed:
        return {tuple(e["qubits"]): e["sign"] < 0 for e in level["target"]["edge_signs"]}
    return {tuple(c["qubits"]): c["value"] < 0 for c in level["results"]["zz_couplings"]}


def sampled(level, rank=0):
    """The maze one measured shot draws: neighbours whose bits differ are split by a wall (rank 0: the most frequent)."""
    bits = level["results"]["measurements"][rank]["bitstring"]
    return {tuple(c["qubits"]): bits[c["qubits"][0]] != bits[c["qubits"][1]] for c in level["results"]["zz_couplings"]}


def mazes(level, views, size=2.6):
    """Draw {label: walls} for one level, side by side: rooms are cells, walls are thick lines."""
    rows, cols = level["grid_size"]["rows"], level["grid_size"]["cols"]
    fig, axes = plt.subplots(1, len(views), figsize=(size * len(views) * cols / max(rows, cols) + 0.3, size), squeeze=False)
    for ax, (label, wall) in zip(axes[0], views.items()):
        ax.add_patch(plt.Rectangle((0, 0), cols, rows, fill=False, lw=3))
        for (a, b), is_wall in wall.items():
            (ra, ca), (rb, cb) = divmod(a, cols), divmod(b, cols)
            if not is_wall:
                continue
            if ra == rb:  # side by side: a vertical wall between them
                x = max(ca, cb)
                ax.plot([x, x], [rows - ra - 1, rows - ra], "k", lw=3)
            else:         # one above the other: a horizontal wall
                y = rows - max(ra, rb)
                ax.plot([ca, ca + 1], [y, y], "k", lw=3)
        ax.set_xlim(-0.1, cols + 0.1)
        ax.set_ylim(-0.1, rows + 0.1)
        ax.set_aspect("equal")
        ax.set_axis_off()
        ax.set_title(label, fontsize=9)
    plt.show()


def compare(level):
    """How the measured walls match the design: matching edges, corridors that closed, walls that opened."""
    d, m = walls(level, designed=True), walls(level)
    closed = sum(m[e] and not d[e] for e in d)
    opened = sum(d[e] and not m[e] for e in d)
    return f"{sum(d[e] == m[e] for e in d)}/{len(d)} edges as designed, {closed} corridors closed, {opened} walls opened"

## Try it

`level_2_final_miami.json` beside this notebook is a 6 × 6 level from the Backrooms Labyrinth project, with the result of a run on IBM's Miami chip saved in it. 36 rooms need 36 qubits, more than the emulator takes, so this runs its top-left 4 × 5 corner (20 qubits, about 15 seconds).

In [ ]:
full = json.load(open("level_2_final_miami.json"))
level = crop(full, rows=4, cols=5)

job = client.jobs.run(
    engine=ENGINE,
    params={
        "level_data": level,
        "shots": 4096,
        "steps": 3,
        "mode": "emu",
    },
)
out = job.result["output"]
mazes(out, {"designed": walls(out, designed=True), "measured": walls(out), "most frequent shot": sampled(out)})
print(compare(out))

The full level as IBM Miami measured it (saved in the file; no new run):

In [ ]:
mazes(full, {"IBM Miami, 6 × 6": walls(full)}, size=3.2)
print({k: full["metrics"][k] for k in ("num_shots", "steps", "fraction", "depth_2q")})

## Parameters

What to expect from each:

- `level_data`: the level: `grid_size`, `num_qubits` (rows × cols), `coupling_map` (which rooms neighbour), `initial_states` (each room's starting Bloch vector and game flags such as `radiating`, carried through to the output) and `relationships`.
- `shots` (up to 10,000): how many times the level is measured. More shots, steadier walls.
- `steps` (1 to 5): how many sweeps build up the correlations before measuring.
- `fraction`: how strongly each sweep ties neighbours together, as a fraction of a full rotation (default 1/3).
- `k` (1 to 4): how many connected rooms each tracked correlation spans. Higher is more faithful and slower.
- `top_n`: how many of the most frequent shots the output keeps (−1 keeps all).
- `mode`: `emu` simulates; `qpu` runs on IBM hardware (your account needs access). Full-size levels need `qpu`.

It returns the level with `target` (the designed edge signs), `results` (`measurements`, `z_expectations`, `zz_couplings`) and `metrics`.

In [ ]:
# client.engines.get(ENGINE)  # its description, params, input and output files, from the API

## How it works

**Design to targets.** Each edge between neighbouring rooms gets a target sign: $+1$ for a corridor (the rooms should agree), $-1$ for a wall. Rooms start from the Bloch vectors in `initial_states`.

**Prepare.** `steps` sweeps of $ZZ$ rotations (each a `fraction` of a full turn) pull every coupled pair towards its target correlation, on a quantum graph state spanning all rooms at once, as in Labyrinth Core. Tomography over paths of `k` rooms tracks how the state is doing.

**Read the maze.** The state is measured `shots` times. Two maze readings come out: the measured correlation of each edge ($\langle Z_aZ_b\rangle < 0$ is a wall), and each individual shot, where neighbouring bits that differ are a wall. No two shots are the same maze.

**Noise.** On hardware, noise weakens correlations: a designed wall can vanish, a corridor close. The saved Miami run shows a whole 6 × 6 level shaped that way.

## Experiments

Each experiment changes one parameter and keeps the others fixed, and runs one job per value.

### Shots

With few shots, measured correlations are noisy and walls flicker; with more they settle.

In [ ]:
views = {}
for shots in (128, 1024, 8192):
    print(f"shots={shots}")
    out = client.jobs.run(engine=ENGINE, params={"level_data": level, "shots": shots}, quiet=True).result["output"]
    views[f"{shots} shots"] = walls(out)
    print(" ", compare(out))
mazes(out, views)

### Steps

More sweeps build stronger correlations, so the measured maze moves closer to the design.

In [ ]:
views = {}
for steps in (1, 2, 3, 5):
    print(f"steps={steps}")
    out = client.jobs.run(engine=ENGINE, params={"level_data": level, "steps": steps}, quiet=True).result["output"]
    views[f"{steps} steps"] = walls(out)
    print(" ", compare(out))
mazes(out, views)

### Coupling strength

`fraction` sets how hard each sweep pulls; too little leaves the maze loose, too much can overshoot.

In [ ]:
views = {}
for fraction in (0.15, 0.33, 0.6):
    print(f"fraction={fraction}")
    out = client.jobs.run(engine=ENGINE, params={"level_data": level, "fraction": fraction}, quiet=True).result["output"]
    views[f"fraction {fraction}"] = walls(out)
    print(" ", compare(out))
mazes(out, views)

### The full level on hardware

36 rooms need a real QPU. This cell is not run here: uncomment it if your account has QPU access, and expect a queue.

In [ ]:
# job = client.jobs.run(engine=ENGINE, params={"level_data": crop(full, 6, 6), "shots": 4096, "steps": 2, "mode": "qpu"})
# mazes(job.result["output"], {"designed": walls(job.result["output"], designed=True), "measured": walls(job.result["output"])})

## Questions

**How does a game load this?**

The output is the level JSON with `results` filled in: a game reads `zz_couplings` (or one bitstring from `measurements`) for the walls, and each room's flags from `initial_states`.

**Why do the designed and measured mazes differ on a simulator?**

Correlations are built up in `steps` sweeps of finite strength and estimated from finite shots, and neighbouring targets pull against each other. More steps and shots bring them closer.

**How big can a level be?**

20 rooms (qubits) on the emulator. Larger levels need `mode: "qpu"`.

**What's the Backrooms Labyrinth?**

Moth's quantum maze game, whose levels were measured on IBM hardware; the level here is one of them.